# 🌱 Plant Intelligence Hub - Google Colab Version

## Advanced Plant Monitoring powered by AI Technology

This notebook provides a complete plant monitoring system with:
- 📊 Real-time sensor data visualization
- 📸 Plant image upload and analysis
- 🤖 AI-powered health analysis
- 📈 Historical data trends
- 🎮 Gamification features

---

## 📦 Setup and Installation

First, let's install the required packages.

In [ ]:
# Install required packages
!pip install -q plotly ipywidgets Pillow numpy pandas matplotlib seaborn

print("✅ All packages installed successfully!")

## 📚 Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import datetime, timedelta
from PIL import Image
import io
import base64
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import random
import requests

# Set style
# Set dark theme for better visibility (change to 'default' for light theme)
plt.style.use('dark_background')
sns.set_palette("husl")

print("✅ Libraries imported successfully!")

## 🎲 Generate Sample Sensor Data

In [ ]:
def generate_sensor_data(hours=168):
    """
    Generate realistic sensor data for plant monitoring.
    Default: 168 hours (1 week)
    """
    now = datetime.now()
    timestamps = [now - timedelta(hours=i) for i in range(hours, 0, -1)]
    
    # Base sensor values (optimal conditions)
    base_temp = 24  # Celsius
    base_humidity = 65  # Percentage
    base_soil = 42  # Percentage
    base_light = 850  # Lux
    
    # Variation constants for realistic data
    TEMP_VARIANCE = 2  # Standard deviation for temperature noise
    TEMP_DAILY_AMPLITUDE = 3  # Daily temperature swing (±3°C)
    TEMP_CYCLE_HOURS = 12  # Temperature cycle period
    
    HUMIDITY_VARIANCE = 3  # Standard deviation for humidity noise
    HUMIDITY_DAILY_AMPLITUDE = 5  # Daily humidity swing (±5%)
    HUMIDITY_CYCLE_HOURS = 24  # Humidity cycle period
    
    SOIL_VARIANCE = 2  # Standard deviation for soil moisture noise
    SOIL_DECAY_RATE = 0.05  # Moisture decrease per hour (simulates water consumption)
    
    LIGHT_VARIANCE = 100  # Standard deviation for light noise
    LIGHT_DAILY_AMPLITUDE = 400  # Peak sunlight intensity above base
    LIGHT_CYCLE_HOURS = 24  # Day/night cycle
    
    data = {
        'timestamp': timestamps,
        'temperature': [base_temp + np.random.normal(0, TEMP_VARIANCE) + TEMP_DAILY_AMPLITUDE*np.sin(i/TEMP_CYCLE_HOURS) for i in range(hours)],
        'humidity': [base_humidity + np.random.normal(0, HUMIDITY_VARIANCE) + HUMIDITY_DAILY_AMPLITUDE*np.cos(i/HUMIDITY_CYCLE_HOURS) for i in range(hours)],
        'soil_moisture': [base_soil + np.random.normal(0, SOIL_VARIANCE) - SOIL_DECAY_RATE*i for i in range(hours)],
        'light_intensity': [max(0, base_light + np.random.normal(0, LIGHT_VARIANCE) + LIGHT_DAILY_AMPLITUDE*np.sin(i/LIGHT_CYCLE_HOURS*np.pi)) for i in range(hours)]
    }
    
    df = pd.DataFrame(data)
    return df

# Generate data
sensor_df = generate_sensor_data()
print(f"✅ Generated {len(sensor_df)} sensor readings")
print(f"\nLatest readings:")
print(sensor_df.tail(1)[['temperature', 'humidity', 'soil_moisture', 'light_intensity']])

## 📊 Sensor Data Cards

Real-time monitoring of key plant health metrics.

In [ ]:
def create_sensor_cards():
    """
    Create beautiful sensor data cards showing current values
    """
    # Get latest values
    latest = sensor_df.iloc[-1]
    
    # Create subplots for each sensor
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('🌡️ Temperature', '💧 Humidity', '🌱 Soil Moisture', '☀️ Light Intensity'),
        specs=[[{'type': 'indicator'}, {'type': 'indicator'}],
               [{'type': 'indicator'}, {'type': 'indicator'}]],
        vertical_spacing=0.15,
        horizontal_spacing=0.1
    )
    
    # Temperature
    fig.add_trace(go.Indicator(
        mode="gauge+number+delta",
        value=latest['temperature'],
        domain={'x': [0, 1], 'y': [0, 1]},
        title={'text': "°C"},
        delta={'reference': 24},
        gauge={
            'axis': {'range': [None, 40]},
            'bar': {'color': "#ef4444"},
            'threshold': {
                'line': {'color': "red", 'width': 4},
                'thickness': 0.75,
                'value': 30
            }
        }
    ), row=1, col=1)
    
    # Humidity
    fig.add_trace(go.Indicator(
        mode="gauge+number+delta",
        value=latest['humidity'],
        domain={'x': [0, 1], 'y': [0, 1]},
        title={'text': "%"},
        delta={'reference': 65},
        gauge={
            'axis': {'range': [None, 100]},
            'bar': {'color': "#3b82f6"},
            'threshold': {
                'line': {'color': "blue", 'width': 4},
                'thickness': 0.75,
                'value': 80
            }
        }
    ), row=1, col=2)
    
    # Soil Moisture
    fig.add_trace(go.Indicator(
        mode="gauge+number+delta",
        value=latest['soil_moisture'],
        domain={'x': [0, 1], 'y': [0, 1]},
        title={'text': "%"},
        delta={'reference': 42},
        gauge={
            'axis': {'range': [None, 100]},
            'bar': {'color': "#06b6d4"},
            'threshold': {
                'line': {'color': "orange", 'width': 4},
                'thickness': 0.75,
                'value': 30
            }
        }
    ), row=2, col=1)
    
    # Light Intensity
    fig.add_trace(go.Indicator(
        mode="gauge+number+delta",
        value=latest['light_intensity'],
        domain={'x': [0, 1], 'y': [0, 1]},
        title={'text': "lux"},
        delta={'reference': 850},
        gauge={
            'axis': {'range': [None, 2000]},
            'bar': {'color': "#f59e0b"},
            'threshold': {
                'line': {'color': "yellow", 'width': 4},
                'thickness': 0.75,
                'value': 1000
            }
        }
    ), row=2, col=2)
    
    fig.update_layout(
        height=600,
        showlegend=False,
        paper_bgcolor='rgba(26, 26, 46, 0.5)',
        plot_bgcolor='rgba(10, 10, 15, 0.5)',
        font=dict(color='white', size=12),
        title={
            'text': '📊 Real-time Sensor Monitoring',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 24, 'color': 'white'}
        }
    )
    
    fig.show()

create_sensor_cards()

## 📈 Historical Data Trends

Visualize sensor data over time to identify patterns and trends.

In [ ]:
def plot_historical_data(period='week'):
    """
    Plot historical sensor data with interactive controls
    """
    periods = {
        'day': 24,
        'week': 168,
        'month': 720,
        'quarter': 2160
    }
    
    # Generate data for selected period
    hours = periods.get(period, 168)
    df = generate_sensor_data(hours)
    
    # Create interactive plot
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=('Environmental Conditions', 'Growth Factors'),
        vertical_spacing=0.12,
        row_heights=[0.5, 0.5]
    )
    
    # Temperature and Humidity
    fig.add_trace(
        go.Scatter(
            x=df['timestamp'], 
            y=df['temperature'],
            name='Temperature (°C)',
            line=dict(color='#ef4444', width=3),
            mode='lines'
        ),
        row=1, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=df['timestamp'], 
            y=df['humidity'],
            name='Humidity (%)',
            line=dict(color='#3b82f6', width=3),
            mode='lines'
        ),
        row=1, col=1
    )
    
    # Soil Moisture and Light
    fig.add_trace(
        go.Scatter(
            x=df['timestamp'], 
            y=df['soil_moisture'],
            name='Soil Moisture (%)',
            line=dict(color='#06b6d4', width=3),
            mode='lines'
        ),
        row=2, col=1
    )
    
    fig.add_trace(
        go.Scatter(
            x=df['timestamp'], 
            y=df['light_intensity'],
            name='Light Intensity (lux)',
            line=dict(color='#f59e0b', width=3),
            mode='lines',
            yaxis='y2'
        ),
        row=2, col=1
    )
    
    fig.update_xaxes(title_text="Time", row=2, col=1)
    fig.update_yaxes(title_text="Temperature / Humidity", row=1, col=1)
    fig.update_yaxes(title_text="Soil Moisture (%)", row=2, col=1)
    
    fig.update_layout(
        height=800,
        showlegend=True,
        hovermode='x unified',
        paper_bgcolor='rgba(26, 26, 46, 0.5)',
        plot_bgcolor='rgba(10, 10, 15, 0.5)',
        font=dict(color='white'),
        title={
            'text': f'📈 Historical Trends - {period.capitalize()}',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 24, 'color': 'white'}
        },
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        )
    )
    
    fig.show()

# Create interactive widget for period selection
period_selector = widgets.Dropdown(
    options=['day', 'week', 'month', 'quarter'],
    value='week',
    description='Period:',
    style={'description_width': 'initial'}
)

def on_period_change(change):
    clear_output(wait=True)
    display(period_selector)
    plot_historical_data(change['new'])

period_selector.observe(on_period_change, names='value')
display(period_selector)
plot_historical_data('week')

## 📸 Plant Image Upload

Upload a plant image for AI analysis.

In [ ]:
def upload_plant_image():
    """
    Upload and display plant image
    """
    print("📤 Please select a plant image to upload...")
    uploaded = files.upload()
    
    if uploaded:
        # Get the first uploaded file
        filename = list(uploaded.keys())[0]
        image_data = uploaded[filename]
        
        # Open and display image
        img = Image.open(io.BytesIO(image_data))
        
        # Display with matplotlib
        fig, ax = plt.subplots(1, 1, figsize=(10, 8))
        ax.imshow(img)
        ax.axis('off')
        ax.set_title('📸 Uploaded Plant Image', fontsize=20, color='white', pad=20)
        plt.tight_layout()
        plt.show()
        
        print(f"✅ Image uploaded successfully: {filename}")
        return img
    else:
        print("❌ No image uploaded")
        return None

# Upload button
upload_button = widgets.Button(
    description='📤 Upload Plant Image',
    button_style='success',
    tooltip='Click to upload a plant image',
    icon='upload'
)

output = widgets.Output()

def on_upload_click(b):
    with output:
        clear_output(wait=True)
        global uploaded_image
        uploaded_image = upload_plant_image()

upload_button.on_click(on_upload_click)
display(upload_button, output)

# Initialize
uploaded_image = None

## 🤖 AI Analysis Results

Analyze plant health using AI-powered detection.

In [ ]:
def analyze_plant_health():
    """
    Simulate AI analysis of plant health
    """
    # Simulate analysis
    health_score = random.randint(75, 95)
    
    # Analysis categories
    categories = {
        '🔬 Disease Detection': {
            'status': 'Healthy',
            'severity': 'good',
            'details': 'No diseases detected',
            'confidence': random.randint(85, 99)
        },
        '💧 Water Stress': {
            'status': 'Low',
            'severity': 'warning',
            'details': 'Slightly below optimal',
            'confidence': random.randint(70, 90)
        },
        '🐛 Pest Detection': {
            'status': 'Clear',
            'severity': 'good',
            'details': 'No pests identified',
            'confidence': random.randint(80, 95)
        },
        '✅ Care Recommendations': {
            'status': '3 Actions',
            'severity': 'info',
            'details': 'View detailed care plan',
            'confidence': random.randint(75, 90)
        }
    }
    
    # Create visualization
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=list(categories.keys()),
        specs=[[{'type': 'indicator'}, {'type': 'indicator'}],
               [{'type': 'indicator'}, {'type': 'indicator'}]],
        vertical_spacing=0.15
    )
    
    positions = [(1, 1), (1, 2), (2, 1), (2, 2)]
    colors = {'good': '#10b981', 'warning': '#f59e0b', 'info': '#06b6d4'}
    
    for (category, data), (row, col) in zip(categories.items(), positions):
        fig.add_trace(
            go.Indicator(
                mode="gauge+number",
                value=data['confidence'],
                title={'text': f"{data['status']}<br><span style='font-size:0.8em'>{data['details']}</span>"},
                gauge={
                    'axis': {'range': [0, 100]},
                    'bar': {'color': colors[data['severity']]},
                    'steps': [
                        {'range': [0, 50], 'color': 'rgba(255, 0, 0, 0.2)'},
                        {'range': [50, 75], 'color': 'rgba(255, 165, 0, 0.2)'},
                        {'range': [75, 100], 'color': 'rgba(0, 255, 0, 0.2)'}
                    ],
                    'threshold': {
                        'line': {'color': "white", 'width': 2},
                        'thickness': 0.75,
                        'value': data['confidence']
                    }
                }
            ),
            row=row, col=col
        )
    
    fig.update_layout(
        height=700,
        showlegend=False,
        paper_bgcolor='rgba(26, 26, 46, 0.5)',
        plot_bgcolor='rgba(10, 10, 15, 0.5)',
        font=dict(color='white', size=11),
        title={
            'text': f'🤖 AI Analysis Results - Health Score: {health_score}%',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 24, 'color': 'white'}
        }
    )
    
    fig.show()
    
    # Display recommendations
    print("\n💡 Care Recommendations:")
    recommendations = [
        "💧 Increase watering frequency - Current soil moisture is slightly low",
        "☀️ Ensure adequate sunlight exposure - 6-8 hours daily recommended",
        "🌱 Consider fertilizing - Nutrient boost may improve growth"
    ]
    for rec in recommendations:
        print(f"  • {rec}")
    
    return health_score, categories

# Run analysis
health_score, analysis = analyze_plant_health()

## 🏆 Gamification Dashboard

Track your progress and achievements!

In [ ]:
def create_gamification_dashboard():
    """
    Create a gamification dashboard with achievements and leaderboard
    """
    # Current user stats
    current_points = 2650
    next_level_points = 3000
    current_rank = 3
    
    # Daily missions
    daily_missions = [
        {'task': 'Check soil moisture', 'completed': True, 'points': 50},
        {'task': 'Upload plant photo', 'completed': True, 'points': 100},
        {'task': 'Review AI recommendations', 'completed': False, 'points': 75},
        {'task': 'Water 3 plants', 'completed': False, 'points': 150}
    ]
    
    # Achievements
    achievements = [
        {'name': 'Green Thumb 🌱', 'unlocked': True},
        {'name': 'Early Bird 🌅', 'unlocked': True},
        {'name': 'Perfect Week ⭐', 'unlocked': True},
        {'name': 'Plant Master 🏆', 'unlocked': False},
        {'name': 'Tech Savvy 🤖', 'unlocked': False},
        {'name': 'Consistency 🔥', 'unlocked': False}
    ]
    
    # Leaderboard
    leaderboard = [
        {'rank': 1, 'name': 'Sarah Green', 'score': 2850},
        {'rank': 2, 'name': 'Mike Thompson', 'score': 2720},
        {'rank': 3, 'name': 'You', 'score': 2650, 'highlight': True},
        {'rank': 4, 'name': 'Emily Chen', 'score': 2580},
        {'rank': 5, 'name': 'David Park', 'score': 2450}
    ]
    
    # Create progress indicator
    progress = (current_points / next_level_points) * 100
    
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('🏆 Your Progress', '📋 Daily Missions', '🎖️ Achievements', '🌟 Leaderboard'),
        specs=[
            [{'type': 'indicator'}, {'type': 'table'}],
            [{'type': 'bar'}, {'type': 'table'}]
        ],
        vertical_spacing=0.15,
        horizontal_spacing=0.1
    )
    
    # Progress indicator
    fig.add_trace(
        go.Indicator(
            mode="gauge+number+delta",
            value=current_points,
            domain={'x': [0, 1], 'y': [0, 1]},
            title={'text': f"Level 5 → 6<br>Rank #{current_rank}"},
            delta={'reference': next_level_points, 'increasing': {'color': '#10b981'}},
            gauge={
                'axis': {'range': [0, next_level_points]},
                'bar': {'color': "#a855f7"},
                'steps': [
                    {'range': [0, next_level_points], 'color': 'rgba(168, 85, 247, 0.2)'}
                ],
                'threshold': {
                    'line': {'color': "#06b6d4", 'width': 4},
                    'thickness': 0.75,
                    'value': next_level_points
                }
            }
        ),
        row=1, col=1
    )
    
    # Daily Missions table
    mission_status = ['✅' if m['completed'] else '⏳' for m in daily_missions]
    fig.add_trace(
        go.Table(
            header=dict(
                values=['Status', 'Task', 'Points'],
                fill_color='#a855f7',
                align='left',
                font=dict(color='white', size=12)
            ),
            cells=dict(
                values=[
                    mission_status,
                    [m['task'] for m in daily_missions],
                    [f"+{m['points']}" for m in daily_missions]
                ],
                fill_color=[['#10b981' if m['completed'] else '#374151' for m in daily_missions]],
                align='left',
                font=dict(color='white', size=11)
            )
        ),
        row=1, col=2
    )
    
    # Achievements bar chart
    unlocked_count = sum(1 for a in achievements if a['unlocked'])
    fig.add_trace(
        go.Bar(
            x=[unlocked_count, len(achievements) - unlocked_count],
            y=['Unlocked', 'Locked'],
            orientation='h',
            marker=dict(color=['#10b981', '#374151']),
            text=[f"{unlocked_count}", f"{len(achievements) - unlocked_count}"],
            textposition='inside',
            textfont=dict(color='white', size=14)
        ),
        row=2, col=1
    )
    
    # Leaderboard table
    fig.add_trace(
        go.Table(
            header=dict(
                values=['Rank', 'Player', 'Score'],
                fill_color='#a855f7',
                align='left',
                font=dict(color='white', size=12)
            ),
            cells=dict(
                values=[
                    [f"#{l['rank']}" for l in leaderboard],
                    [l['name'] for l in leaderboard],
                    [f"{l['score']:,}" for l in leaderboard]
                ],
                fill_color=[['#a855f7' if l.get('highlight') else '#1f2937' for l in leaderboard]],
                align='left',
                font=dict(color='white', size=11)
            )
        ),
        row=2, col=2
    )
    
    fig.update_xaxes(showticklabels=False, row=2, col=1)
    fig.update_yaxes(showticklabels=True, row=2, col=1)
    
    fig.update_layout(
        height=800,
        showlegend=False,
        paper_bgcolor='rgba(26, 26, 46, 0.5)',
        plot_bgcolor='rgba(10, 10, 15, 0.5)',
        font=dict(color='white'),
        title={
            'text': '🎮 Gamification Dashboard',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 24, 'color': 'white'}
        }
    )
    
    fig.show()
    
    # Display achievements text
    print("\n🎖️ Your Achievements:")
    for ach in achievements:
        status = "✅" if ach['unlocked'] else "🔒"
        print(f"  {status} {ach['name']}")

create_gamification_dashboard()

## 📊 Complete Dashboard Summary

All-in-one view of your plant monitoring system.

In [ ]:
def create_complete_dashboard():
    """
    Create a comprehensive dashboard with all key metrics
    """
    print("🌱 PLANT INTELLIGENCE HUB - COMPLETE DASHBOARD\n")
    print("=" * 60)
    
    # Latest sensor readings
    latest = sensor_df.iloc[-1]
    print("\n📊 CURRENT SENSOR READINGS:")
    print(f"  🌡️  Temperature:     {latest['temperature']:.1f}°C")
    print(f"  💧 Humidity:        {latest['humidity']:.1f}%")
    print(f"  🌱 Soil Moisture:   {latest['soil_moisture']:.1f}%")
    print(f"  ☀️  Light Intensity: {latest['light_intensity']:.0f} lux")
    
    # Health status
    print("\n🤖 AI HEALTH ANALYSIS:")
    print(f"  Overall Health Score: {health_score}%")
    print(f"  Status: {'✅ Excellent' if health_score >= 85 else '⚠️ Needs Attention'}")
    
    # Alerts
    print("\n⚠️ ACTIVE ALERTS:")
    if latest['soil_moisture'] < 35:
        print("  • LOW SOIL MOISTURE - Consider watering soon")
    if latest['light_intensity'] < 500:
        print("  • LOW LIGHT - Move to brighter location")
    if latest['temperature'] > 28:
        print("  • HIGH TEMPERATURE - Ensure adequate ventilation")
    
    if (latest['soil_moisture'] >= 35 and 
        latest['light_intensity'] >= 500 and 
        latest['temperature'] <= 28):
        print("  ✅ No active alerts - All systems optimal")
    
    # Statistics
    print("\n📈 7-DAY STATISTICS:")
    print(f"  Temperature   - Avg: {sensor_df['temperature'].mean():.1f}°C, "
          f"Min: {sensor_df['temperature'].min():.1f}°C, "
          f"Max: {sensor_df['temperature'].max():.1f}°C")
    print(f"  Humidity      - Avg: {sensor_df['humidity'].mean():.1f}%, "
          f"Min: {sensor_df['humidity'].min():.1f}%, "
          f"Max: {sensor_df['humidity'].max():.1f}%")
    print(f"  Soil Moisture - Avg: {sensor_df['soil_moisture'].mean():.1f}%, "
          f"Min: {sensor_df['soil_moisture'].min():.1f}%, "
          f"Max: {sensor_df['soil_moisture'].max():.1f}%")
    print(f"  Light         - Avg: {sensor_df['light_intensity'].mean():.0f} lux, "
          f"Min: {sensor_df['light_intensity'].min():.0f} lux, "
          f"Max: {sensor_df['light_intensity'].max():.0f} lux")
    
    print("\n" + "=" * 60)
    print("✅ Dashboard generated successfully!")
    print("\nℹ️ Scroll up to see interactive visualizations")

create_complete_dashboard()

## 📖 Usage Guide

### How to Use This Notebook:

1. **Setup**: Run all cells from top to bottom (Runtime → Run all)

2. **View Sensor Data**: 
   - Real-time sensor gauges show current readings
   - Use the period dropdown to view historical trends

3. **Upload Plant Images**: 
   - Click the "Upload Plant Image" button
   - Select an image from your device
   - The image will be displayed for analysis

4. **AI Analysis**: 
   - Automatically analyzes plant health
   - Shows disease detection, water stress, pest identification
   - Provides care recommendations

5. **Track Progress**: 
   - View your achievements and rank
   - Complete daily missions for points
   - Compete on the leaderboard

### Features:
- ✅ Real-time sensor monitoring
- ✅ Interactive historical charts
- ✅ Image upload and analysis
- ✅ AI-powered health assessment
- ✅ Gamification and achievements
- ✅ Comprehensive dashboard

### Notes:
- Data is simulated for demonstration purposes
- In production, connect to real IoT sensors
- AI analysis uses mock data - integrate real ML models for actual plant disease detection
- All visualizations are interactive - hover for details, zoom, pan, etc.

---

**Made with ❤️ for plant lovers and tech enthusiasts!**
